In [ ]:
  import os
import sys
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import euclidean_distances
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
import torch
from torch.utils.data import TensorDataset, DataLoader
from google.colab import drive


In [ ]:
def random_seed(seed=1234):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

random_seed(1234)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Add dataset directory to path
import sys
sys.path.append('/content/drive/MyDrive/Collected_dataset')

# OS and path setup
import os
ospj = os.path.join
path_workspace = '/content/drive/MyDrive/'

Mounted at /content/drive


In [ ]:
# Define training data paths
dict_path_tr_clean = {
    #'20m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_static.csv'),
     '20m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_dyn1.csv'),
    # '20m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_dyn2.csv'),
    #'50m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_static.csv'),
     '50m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_dyn1.csv'),
    # '50m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_dyn2.csv'),
    #'70m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_static.csv'),
     '70m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_dyn1.csv'),
    # '70m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_dyn2.csv'),
}


dict_path_te_clean = {
    # '35m_stat': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_stat.csv'),
     '35m_dyn1': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_dyn1.csv'),
    # '35m_dyn2': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_dyn2.csv'),
}



# dict_path_tr_clean = {
#     '20m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_static.csv'),
#     '20m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_dyn1.csv'),
#     '20m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_dyn2.csv'),
#     '50m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_static.csv'),
#     '50m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_dyn1.csv'),
#     '50m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_dyn2.csv'),
#     '70m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_static.csv'),
#     '70m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_dyn1.csv'),
#     '70m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_dyn2.csv'),
# }


# dict_path_te_clean = {
#     '35m_stat': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_stat.csv'),
#     '35m_dyn1': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_dyn1.csv'),
#     '35m_dyn2': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_dyn2.csv'),
# }






# dict_path_tr_clean = {
#     '35m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/35m_stat_train.csv'),
# }
# dict_path_te_clean = {
#     '35m_stat': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_stat_test.csv'),
# }

# 우선
# 20 ,50 ,70 stat 만 file , 1 개씩 valid 35 stat test 로 파일 구성
# training 하고 test 는 다를수있어도 valid 랑 test 는 유사한지 확인
# baseline 모델 이랑 lstm 구현 epoch 은 100 만 수요일 4시반 목요일대신





In [ ]:
import pandas as pd

# Select specific columns
selected_columns = ['lat', 'lon', 'alt', 'alt_ellipsoid','vel_m_s', 'vel_n_m_s', 'vel_e_m_s', 'vel_d_m_s', 'cog_rad', 'timestamp']

# Initialize dictionaries
dict_df_tr_clean = {}
dict_df_te_clean = {}

# Load training CSVs
for key, csv_path in dict_path_tr_clean.items():
    try:
        df = pd.read_csv(csv_path, usecols=selected_columns)
        dict_df_tr_clean[key] = df
        print(f"{key} ✅ Training data loaded")
    except Exception as e:
        print(f"{key} ❌ Training load error: {e}")

# Load testing CSVs
for key, csv_path in dict_path_te_clean.items():
    try:
        df = pd.read_csv(csv_path, usecols=selected_columns)
        dict_df_te_clean[key] = df
        print(f"{key} ✅ Test data loaded")
    except Exception as e:
        print(f"{key} ❌ Test load error: {e}")


20m_dyn1 ✅ Training data loaded
50m_dyn1 ✅ Training data loaded
70m_dyn1 ✅ Training data loaded
35m_dyn1 ✅ Test data loaded


In [ ]:
def convert_timestamp_to_seconds(t):
    try:
        minutes, sec_fraction = t.split(":")
        return int(minutes) * 60 + float(sec_fraction)
    except Exception as e:
        print(f"Error converting timestamp: {t} -> {e}")
        return None


In [ ]:
def split_by_time_gap(df, time_col='timestamp', threshold=1.0):
    df = df.copy()
    df[time_col] = df[time_col].apply(convert_timestamp_to_seconds)
    df = df.dropna(subset=[time_col]).reset_index(drop=True)

    print("\n🧪 Converted timestamp (head):")
    print(df[time_col].iloc[:5].tolist())

    diffs = df[time_col].diff().fillna(0).abs()
    split_indices = diffs[diffs >= threshold].index.tolist()

    split_indices = [0] + split_indices + [len(df)]
    splits = [
        df.iloc[split_indices[i]:split_indices[i + 1]].reset_index(drop=True)
        for i in range(len(split_indices) - 1)
    ]
    return splits


In [ ]:
# Apply segment split to training and testing data
dict_split_tr_clean = {}
dict_split_te_clean = {}

for key, df in dict_df_tr_clean.items():
    split_dfs = split_by_time_gap(df, time_col='timestamp', threshold=1.0)
    dict_split_tr_clean[key] = split_dfs
    print(f"{key} ✅ Segmented - {len(split_dfs)} segments")

for key, df in dict_df_te_clean.items():
    split_dfs = split_by_time_gap(df, time_col='timestamp', threshold=1.0)
    dict_split_te_clean[key] = split_dfs
    print(f"{key} ✅ Segmented - {len(split_dfs)} segments")



🧪 Converted timestamp (head):
[270.4, 270.6, 270.8, 271.0, 271.2]
20m_dyn1 ✅ Segmented - 4 segments

🧪 Converted timestamp (head):
[256.0, 256.2, 256.4, 256.6, 256.8]
50m_dyn1 ✅ Segmented - 4 segments

🧪 Converted timestamp (head):
[330.6, 330.8, 331.0, 331.2, 331.4]
70m_dyn1 ✅ Segmented - 5 segments

🧪 Converted timestamp (head):
[226.7, 226.9, 227.1, 227.3, 227.5]
35m_dyn1 ✅ Segmented - 4 segments


In [ ]:
dict_split_train_only = {}
dict_split_valid_only = {}

for key, segments in dict_split_tr_clean.items():
    if len(segments) > 1:
        dict_split_train_only[key] = segments[:-1]
        dict_split_valid_only[key] = [segments[-1]]
    else:
        dict_split_train_only[key] = []
        dict_split_valid_only[key] = segments

# 확인
for k in dict_split_tr_clean.keys():
    print(f"{k} → Train seg: {len(dict_split_train_only[k])}, Valid seg: {len(dict_split_valid_only[k])}")

20m_dyn1 → Train seg: 3, Valid seg: 1
50m_dyn1 → Train seg: 3, Valid seg: 1
70m_dyn1 → Train seg: 4, Valid seg: 1


In [ ]:
def count_total_points(dict_original, dict_split):
    for key in dict_original:
        original_count = len(dict_original[key])
        split_count = sum(len(df) for df in dict_split[key])
        status = "✅ 일치" if original_count == split_count else "❌ 불일치"
        print(f"{key}: 원본 = {original_count} / 분할 합계 = {split_count} → {status}")




In [ ]:

# =====================================
# 3. 원본 vs Segment 개수 비교
# =====================================
print("\n🔎 학습 데이터 확인 (원본 vs 전체 segment)")
count_total_points(dict_df_tr_clean, dict_split_tr_clean)

print("\n🔎 테스트 데이터 확인 (원본 vs 전체 segment)")
count_total_points(dict_df_te_clean, dict_split_te_clean)



🔎 학습 데이터 확인 (원본 vs 전체 segment)
20m_dyn1: 원본 = 1826 / 분할 합계 = 1826 → ✅ 일치
50m_dyn1: 원본 = 1980 / 분할 합계 = 1980 → ✅ 일치
70m_dyn1: 원본 = 2000 / 분할 합계 = 2000 → ✅ 일치

🔎 테스트 데이터 확인 (원본 vs 전체 segment)
35m_dyn1: 원본 = 2120 / 분할 합계 = 2120 → ✅ 일치


In [ ]:
# =====================================
# 4. Train / Valid 포인트 수 요약
# =====================================
train_points = sum(len(df) for dfs in dict_split_train_only.values() for df in dfs)
valid_points = sum(len(df) for dfs in dict_split_valid_only.values() for df in dfs)
test_points  = sum(len(df) for dfs in dict_split_te_clean.values()   for df in dfs)

print("\n📊 데이터 포인트 개수 요약")
print(f"Train total points: {train_points}")
print(f"Valid total points: {valid_points}")
print(f"Test total points:  {test_points}")


📊 데이터 포인트 개수 요약
Train total points: 4444
Valid total points: 1362
Test total points:  2120


In [ ]:
def add_delta_coordinates(dict_split):
    for key, split_list in dict_split.items():
        for df in split_list:
            df['D_lat'] = [0.0] + (df['lat'].diff().fillna(0).tolist())[1:]
            df['D_lon'] = [0.0] + (df['lon'].diff().fillna(0).tolist())[1:]
            df['D_alt'] = [0.0] + (df['alt'].diff().fillna(0).tolist())[1:]


In [ ]:
from sklearn.metrics import euclidean_distances

def calculate_euc_distance(df):
    coordinates = df[['lat', 'lon', 'alt']].to_numpy()
    distances = [0.0]
    for i in range(1, len(coordinates)):
        dist = euclidean_distances([coordinates[i - 1]], [coordinates[i]])[0][0]
        distances.append(dist)
    return distances

def add_euc_distance(dict_split):
    for key, split_list in dict_split.items():
        for df in split_list:
            distances = calculate_euc_distance(df)
            df['d_pos'] = distances


In [ ]:
def add_time_gap(dict_split):
    for key, split_list in dict_split.items():
        for df in split_list:
            time_gap = [0.0] + (df['timestamp'].diff().fillna(0).tolist())[1:]
            df['time_gap'] = time_gap


In [ ]:
# Apply feature creation to both training and test sets
add_delta_coordinates(dict_split_train_only)
add_delta_coordinates(dict_split_valid_only)
add_delta_coordinates(dict_split_te_clean)

add_euc_distance(dict_split_train_only)
add_euc_distance(dict_split_valid_only)
add_euc_distance(dict_split_te_clean)

add_time_gap(dict_split_train_only)
add_time_gap(dict_split_valid_only)
add_time_gap(dict_split_te_clean)


In [ ]:
# =====================================
# 세트별 segment 개수 더블 체크
# =====================================
print("\n📦 Segment 개수 확인")
print(f"Train set: {sum(len(segments) for segments in dict_split_train_only.values())} segments total")
print(f"Valid set: {sum(len(segments) for segments in dict_split_valid_only.values())} segments total")
print(f"Test set:  {sum(len(segments) for segments in dict_split_te_clean.values())} segments total")

# key별 segment 개수 확인 (원하면 주석 해제)
for k in dict_split_train_only.keys():
    print(f"{k} → Train seg: {len(dict_split_train_only[k])}, Valid seg: {len(dict_split_valid_only[k])}")



📦 Segment 개수 확인
Train set: 10 segments total
Valid set: 3 segments total
Test set:  4 segments total
20m_dyn1 → Train seg: 3, Valid seg: 1
50m_dyn1 → Train seg: 3, Valid seg: 1
70m_dyn1 → Train seg: 4, Valid seg: 1


In [ ]:
# # Extract rows where time_gap < 0
# neg_time_gap_rows = dict_split_tr_clean['20m_stat'][0][dict_split_tr_clean['20m_stat'][0]['time_gap'] < 0]

# # Display them
# print("❗ Rows with time_gap < 0:")
# print(neg_time_gap_rows)


In [ ]:
import numpy as np

# Define window size
window_size = 5 # 기본 5개
# Real 5개를 가지고 6번째 예측
#2,3,4,5,6 을보고 7을 예측

# Define input features and target features
input_features =  ['vel_m_s', 'vel_n_m_s', 'vel_e_m_s', 'cog_rad', 'D_lat', 'D_lon', 'd_pos','time_gap']
# 'vel_m_s', 'vel_n_m_s', 'vel_e_m_s', 'vel_d_m_s', 'cog_rad', 'D_lat', 'D_lon', 'D_alt', 'd_pos','time_gap'

target_features = ['D_lon']  # target to predict


In [ ]:
def create_sequence_label_tensors(dict_split, window_size):
    X_all, y_all = [], []
    for df_list in dict_split.values():
        for df in df_list:
            if len(df) <= window_size:
                continue
            data = df[input_features].to_numpy()
            labels = df[target_features].to_numpy()
            for i in range(len(df) - window_size):
                window = data[i : i + window_size]
                label = labels[i + window_size]
                X_all.append(window)
                y_all.append(label)
    X_tensor = np.array(X_all)
    y_tensor = np.array(y_all)
    return X_tensor, y_tensor

In [ ]:
# Apply sequence generation
X_train, y_train = create_sequence_label_tensors(dict_split_train_only, window_size)
X_valid, y_valid = create_sequence_label_tensors(dict_split_valid_only, window_size)
X_test,  y_test  = create_sequence_label_tensors(dict_split_te_clean,  window_size)


In [ ]:
# Check resulting shapes
print(f"✅ Train sequences shape: {X_train.shape}")
print(f"✅ Train labels shape:    {y_train.shape}")

print(f"✅ Valid sequences shape: {X_valid.shape}")
print(f"✅ Valid labels shape:    {y_valid.shape}")

print(f"✅ Test sequences shape:  {X_test.shape}")
print(f"✅ Test labels shape:     {y_test.shape}")

✅ Train sequences shape: (4394, 5, 8)
✅ Train labels shape:    (4394, 1)
✅ Valid sequences shape: (1347, 5, 8)
✅ Valid labels shape:    (1347, 1)
✅ Test sequences shape:  (2100, 5, 8)
✅ Test labels shape:     (2100, 1)


In [ ]:
def count_expected_windows(dict_split, window_size):
    expected_total = 0
    for df_list in dict_split.values():
        for df in df_list:
            expected_total += max(0, len(df) - window_size)
    return expected_total

# ✅ 세트별 기대 윈도우 개수
expected_train = count_expected_windows(dict_split_train_only, window_size)
expected_valid = count_expected_windows(dict_split_valid_only, window_size)
expected_test  = count_expected_windows(dict_split_te_clean,  window_size)

# ✅ 실제 생성된 윈도우 개수 (앞서 만든 넘파이 배열 기준)
actual_train = X_train.shape[0]
actual_valid = X_valid.shape[0]
actual_test  = X_test.shape[0]

# ✅ 매칭 여부 출력
print(f"✅ Train: expected = {expected_train}, actual = {actual_train} → {'✔️ Match' if expected_train == actual_train else '❌ Mismatch'}")
print(f"✅ Valid: expected = {expected_valid}, actual = {actual_valid} → {'✔️ Match' if expected_valid == actual_valid else '❌ Mismatch'}")
print(f"✅ Test:  expected = {expected_test},  actual = {actual_test}  → {'✔️ Match' if expected_test  == actual_test  else '❌ Mismatch'}")


✅ Train: expected = 4394, actual = 4394 → ✔️ Match
✅ Valid: expected = 1347, actual = 1347 → ✔️ Match
✅ Test:  expected = 2100,  actual = 2100  → ✔️ Match


In [ ]:
def create_sequence_label_tensors(dict_split, window_size):
    X_all, y_all = [], []
    for df_list in dict_split.values():
        for df in df_list:
            if len(df) <= window_size:
                continue
            data = df[input_features].to_numpy()
            labels = df[target_features].to_numpy()
            for i in range(len(df) - window_size):
                window = data[i : i + window_size]
                label = labels[i + window_size]
                X_all.append(window)
                y_all.append(label)
    X_tensor = np.array(X_all)
    y_tensor = np.array(y_all)
    return X_tensor, y_tensor

In [ ]:
X_train, y_train = create_sequence_label_tensors(dict_split_train_only, window_size)
X_valid, y_valid = create_sequence_label_tensors(dict_split_valid_only, window_size)
X_test,  y_test  = create_sequence_label_tensors(dict_split_te_clean,  window_size)


In [ ]:
print(f"✅ Train sequences shape: {X_train.shape}")
print(f"✅ Train labels shape:    {y_train.shape}")
print(f"✅ Valid sequences shape: {X_valid.shape}")
print(f"✅ Valid labels shape:    {y_valid.shape}")
print(f"✅ Test sequences shape:  {X_test.shape}")
print(f"✅ Test labels shape:     {y_test.shape}")

✅ Train sequences shape: (4394, 5, 8)
✅ Train labels shape:    (4394, 1)
✅ Valid sequences shape: (1347, 5, 8)
✅ Valid labels shape:    (1347, 1)
✅ Test sequences shape:  (2100, 5, 8)
✅ Test labels shape:     (2100, 1)


In [ ]:
def make_loader(X, y, batch_size=64, shuffle=True):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32)
    dataset = TensorDataset(X_tensor, y_tensor)

    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [ ]:
# Train loader
train_loader = make_loader(X_train, y_train, batch_size=64, shuffle=True)

# Valid loader
X_val_tensor = torch.tensor(X_valid, dtype=torch.float32)
y_val_tensor = torch.tensor(y_valid, dtype=torch.float32)
valid_dataset = TensorDataset(X_val_tensor, y_val_tensor)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)

# Test loader
X_te_tensor = torch.tensor(X_test, dtype=torch.float32)
y_te_tensor = torch.tensor(y_test, dtype=torch.float32)
test_dataset = TensorDataset(X_te_tensor, y_te_tensor)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
import numpy as np
import torch
from sklearn.metrics import mean_squared_error

def compute_all_metrics(preds, targets):
    if isinstance(preds, torch.Tensor):
        preds = preds.detach().cpu().numpy()
    if isinstance(targets, torch.Tensor):
        targets = targets.detach().cpu().numpy()

    mse = mean_squared_error(targets, preds)
    return mse



In [ ]:
def show_delta_predictions(preds, targets, n=10):
    if isinstance(preds, torch.Tensor):
        preds = preds.detach().cpu().numpy()
    if isinstance(targets, torch.Tensor):
        targets = targets.detach().cpu().numpy()
    df = pd.DataFrame({
        'D_lon_pred': preds[:n],
        'D_lon_true': targets[:n],
    })
    return df.round(4)


In [ ]:
# 1) 3D -> 2D로 펼치기
# X만 (N, window, feature) → (N, window*feature) 로 변환
def to_2d(X):
    return X.reshape(X.shape[0], -1)

X_train_2d = to_2d(X_train)
X_valid_2d = to_2d(X_valid)
X_test_2d  = to_2d(X_test)

# (N,1) → (N,)
y_train_1d = y_train.reshape(-1)
y_valid_1d = y_valid.reshape(-1)
y_test_1d  = y_test.reshape(-1)






In [ ]:
import pandas as pd
models = {
    'LinearRegression': LinearRegression(),
    'SVR': SVR(),
    'RandomForest': RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
}

for name, model in models.items():
    print(f"\n=== {name} ===")
    model.fit(X_train_2d, y_train_1d)

    # Train
    y_pred_tr = model.predict(X_train_2d)
    tr_mse = compute_all_metrics(y_pred_tr, y_train_1d)
    print(f"Train MSE: {tr_mse:.6f}")

    # Valid
    y_pred_va = model.predict(X_valid_2d)
    va_mse = compute_all_metrics(y_pred_va, y_valid_1d)
    print(f"Valid MSE: {va_mse:.6f}")

    # Test
    y_pred_te = model.predict(X_test_2d)
    te_mse = compute_all_metrics(y_pred_te, y_test_1d)
    print(f"Test  MSE: {te_mse:.6f}")

    # 앞 10개 비교 (함수 활용)
    df_compare = show_delta_predictions(y_pred_te, y_test_1d, n=10)
    print(df_compare)

    # 간단 요약 통계
    df_te = pd.DataFrame({
        "true_D_lon": y_test_1d,
        "pred_D_lon": y_pred_te
    })
    df_te["error_D_lon"] = df_te["pred_D_lon"] - df_te["true_D_lon"]
    print("\n📋 D_lon Summary (true, pred, error)")
    print(df_te[["true_D_lon","pred_D_lon","error_D_lon"]].describe().round(6))


=== LinearRegression ===
Train MSE: 7.691991
Valid MSE: 8.187238
Test  MSE: 5.092851
   D_lon_pred  D_lon_true
0    -12.7507       -14.0
1    -14.7858       -14.0
2    -14.5762       -17.0
3    -18.6742       -16.0
4    -17.1335       -15.0
5    -15.6454       -10.0
6    -12.9788       -13.0
7    -11.0370       -12.0
8     -9.4186        -7.0
9     -5.3173        -7.0

📋 D_lon Summary (true, pred, error)
        true_D_lon   pred_D_lon  error_D_lon
count  2100.000000  2100.000000  2100.000000
mean      0.914286     0.897910    -0.016376
std      22.791080    22.782829     2.257213
min     -51.000000   -52.701304   -13.910917
25%      -5.000000    -5.595904    -1.288579
50%       0.000000     0.171672    -0.004628
75%       9.000000     9.158720     1.229330
max      46.000000    46.314199    12.040100

=== SVR ===
Train MSE: 67.057317
Valid MSE: 29.444669
Test  MSE: 12.239881
   D_lon_pred  D_lon_true
0    -10.3639       -14.0
1    -11.2815       -14.0
2    -11.7612       -17.0
3    -